In [1]:
import os
import httpx
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env")
STRIPE_API_KEY = os.getenv("STRIPE_TEST_KEY")

PRODUCT_CATALOG = {
    "PROD_001": {
        "name": "Enterprise AI Workstation",
        "price_cents": 120000,
        "currency": "usd",
    },
    "PROD_002": {
        "name": "Developer Cloud Credit Pack",
        "price_cents": 5000,
        "currency": "usd",
    },
}

def process_catalog_checkout(product_id: str, customer_email: str):
    product = PRODUCT_CATALOG.get(product_id)
    if not product:
        return {"status":"FAILED", "reason": f"Product {product_id} not found"}

    url_customer = "https://api.stripe.com/v1/customers"
    url_pi = "https://api.stripe.com/v1/payment_intents"

    headers = {
        "Authorization":f"Bearer {STRIPE_API_KEY}",
        "Content_Type": "application/x-www-form-urlencoded",
    }

    cust_payload = {
        "email": customer_email,
        "description": f"Customer for catalog order {product_id}",
    }

    cust_response = httpx.post(
        url_customer, headers=headers, data=cust_payload
    )

    if cust_response.status_code != 200:
        return {
            "status": "FAILED",
            "step": "Customer Creation",
            "error": cust_response.text,
        }

    customer_id = cust_response.json().get("id")

    pi_payload = {
        "amount": str(product.get("price_cents")),
        "currency":product.get("currency"),
        "customer":customer_id,
        "description":f"Purchasse: {product.get('name')}",
    }

    pi_response = httpx.post(url_pi, headers=headers, data=pi_payload)

    if pi_response.status_code == 200:
        pi_data = pi_response.json()
        return {
            "status": "SUCCESS",
            "order_summary": {
                "product_name": product.get("name"),
                "amount_usd": product.get("price_cents") / 100,
                "customer_id": customer_id,
                "payment_intent_id": pi_data.get("id"),
                "payment_status": pi_data.get("status"),
                "client_secret": pi_data.get("client_secret"),
            },
        }
    else:
        return {
            "status": "FAILED",
            "step": "PaymentIntent Creation",
            "error": pi_response.text,
        }

order_result = process_catalog_checkout(
    product_id="PROD_001", customer_email="enterprise.buyer@example.com"
)

print("Product Catalog Checkout Execution:")
print(order_result)
            

Product Catalog Checkout Execution:
{'status': 'SUCCESS', 'order_summary': {'product_name': 'Enterprise AI Workstation', 'amount_usd': 1200.0, 'customer_id': 'cus_VVqcW8Ojn4zQps', 'payment_intent_id': 'pi_3UUovd2XfG9T4MP71HUm7ntN', 'payment_status': 'requires_payment_method', 'client_secret': 'pi_3UUovd2XfG9T4MP71HUm7ntN_secret_PHzJHKGqX7sb11dtA6M5UdhE4'}}


In [9]:
def parse_payment_event(event_dict):
    return {
        "event_type": event_dict.get("type"),
        "payment_intent_id": event_dict.get("data",{}).get("object",{}).get("id"),
    }
   
    
mock_stripe_event = {
    "id": "evt_3Mv1234567890",
    "object": "event",
    "type": "payment_intent.succeeded",
    "data": {
        "object": {
            "id": "pi_9988776655",
            "amount": 4500,  # in cents ($45.00)
            "currency": "sgd",
            "customer": "cus_N1A2B3C4D5",
            "receipt_email": "alex.dev@example.com",
            "status": "succeeded",
        }
    },
}

hook = parse_payment_event(mock_stripe_event)
print("Parsed Event Summary:")
print(hook)

Parsed Event Summary:
{'event_type': 'payment_intent.succeeded', 'payment_intent_id': 'pi_9988776655'}
